# Data Preprocessing
Preprocess `cross_references.tsv` according to the initial EDA.

In [1]:
import pandas as pd

In [61]:
raw_data = pd.read_csv('cross_references.tsv', sep='\t')
raw_data.sample(10)

,From Verse,To Verse,Votes,#www.openbible.info CC-BY 2026-09-07
49069,Josh.10.24,Mal.4.3,6,NaN
5541,Gen.22.6,Matt.8.17,1,NaN
286244,Acts.24.3,Luke.1.3,3,NaN
233647,Matt.8.3,2Kgs.5.11,6,NaN
241102,Matt.24.6,Matt.24.14,7,NaN
48160,Josh.7.2,Hos.4.15,4,NaN
116182,Ps.35.16,Lam.2.16,6,NaN
200550,Ezek.20.38,Matt.3.9-Matt.3.10,0,NaN
225252,Zech.1.4,2Chr.36.15-2Chr.36.16,3,NaN
269976,John.12.21,John.1.43-John.1.47,3,NaN


In [62]:
data = raw_data.iloc[:,0:3]
display(data.shape)
display(data.sample(5))

(344756, 3)

,From Verse,To Verse,Votes
35084,Num.21.25,Isa.16.8-Isa.16.9,2
29981,Lev.26.43,Dan.9.7-Dan.9.9,2
148236,Prov.28.5,1John.2.20,10
30168,Lev.27.34,Deut.4.5,2
40047,Deut.7.16,Deut.13.8,7


In [63]:
# remove negative and zero votes
data_positive = data.loc[data['Votes']>0]
display(data_positive.shape)
display(data_positive.sample(5))

(341242, 3)

,From Verse,To Verse,Votes
83924,1Chr.17.22,Rom.11.1-Rom.11.12,2
172398,Isa.60.6,Isa.42.10,5
203208,Ezek.28.17,Ezek.32.10,3
145308,Prov.18.9,Heb.6.12,12
149607,Eccl.1.16,1Kgs.10.23-1Kgs.10.24,2


In [96]:
def filter_func(row, max_verse):
    # identify if the from verse and to verse are from the same chapter
    chap1 = '.'.join(row['From Verse'].split('.')[:2])
    chap2 = '.'.join(row['To Verse'].split('.')[:2])
    same_chapter = chap1 == chap2

    # if the To Verse spans multiple verses:
    if '-' in row['To Verse']:
        # Check whether it spans multiple chapters
        start, end = row['To Verse'].split('-')
        start_parts = start.split('.')
        end_parts = end.split('.')

        # check whether 'To Verse' stays within one chapter
        not_multi_chapter = start_parts[:2] == end_parts[:2]
        
        # check the number of verses it spans
        start_verse = int(start_parts[2])
        end_verse = int(end_parts[2])
        verse_span = end_verse - start_verse + 1
    else:
        not_multi_chapter = True
        verse_span = 1
    return same_chapter and not_multi_chapter and (verse_span <= max_verse)

In [102]:
# Apply filter
idx_filter = data_positive.apply(filter_func, max_verse=5, axis=1) # set max verse span to 5
data_filtered = data_positive.loc[idx_dif, :]
display(data_filtered.shape)
display(data_filtered.sample(10))

(326215, 3)

,From Verse,To Verse,Votes
202717,Ezek.27.5,1Kgs.5.1,2
215211,Hos.9.17,Prov.29.1,2
9219,Gen.35.6,Gen.48.3,2
335935,1John.2.13,John.14.9,4
198972,Ezek.16.56,Luke.15.28-Luke.15.30,2
267880,John.8.44,1Chr.21.1,11
99225,Esth.9.22,Esth.3.12-Esth.3.13,3
216089,Hos.13.15,Hos.10.1,2
267132,John.7.39,Eph.4.30,4
249865,Mark.15.2,John.18.29-John.18.38,2


In [ ]:
# save filtered data
data_filtered.to_csv('